# Quotation Win/Loss Prediction — Decision Tree

Model 2 of 7. Uses the shared `quotation_pipeline_utils.py` (same file notebook 1 uses) for cleaning — one `QuotationCleaner`, one place the logic lives, imported here rather than redefined. See notebook 1 for the full EDA and the reasoning behind each cleaning decision; this one focuses on what's different for a tree.

> ⚠️ `quotation_pipeline_utils.py` must be in the same folder as this notebook. Update `DATA_PATH` below.


In [ ]:
%pip install shap -q


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix,
                              classification_report, RocCurveDisplay, PrecisionRecallDisplay,
                              brier_score_loss)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

from quotation_pipeline_utils import (
    QuotationCleaner, filter_valid_rows, select_binary_features,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES
)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Imported QuotationCleaner from quotation_pipeline_utils.py')


In [ ]:
DATA_PATH = '/mnt/user-data/uploads/Quotation_Data.xlsx'   # <-- update this to your local path

RANDOM_STATE = 42
TEST_FRACTION = 0.20
MIN_CATEGORY_COUNT = 15
RARE_WALLTYPE_PCT = 1.0


## Load & split (identical logic to notebook 1, via the shared cleaner)

In [ ]:
df_raw = pd.read_excel(DATA_PATH, sheet_name='Data')
df_valid = filter_valid_rows(df_raw)
df_valid['_Date_parsed'] = pd.to_datetime(df_valid['Date'], errors='coerce')
df_sorted = df_valid.sort_values('_Date_parsed').reset_index(drop=True)

TARGET = 'Success'
X = df_sorted.drop(columns=[TARGET, '_Date_parsed'])
y = df_sorted[TARGET]
split_idx = int(len(df_sorted) * (1 - TEST_FRACTION))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print('Train:', len(X_train), '| Test:', len(X_test), '| Split date:', df_sorted['_Date_parsed'].iloc[split_idx].date())


## Encoding: target encoding, not one-hot

Same conclusion as the original version of this notebook, summarized rather than re-derived in full: one-hot forces a shallow, interpretability-constrained tree to spend its limited splits distinguishing one category at a time — inefficient for `Client_Clean` (808 categories) and `Suburb` (213). Compared directly (one-hot / ordinal / target encoding, each tuned separately): ordinal scored highest (ROC-AUC 0.62) but produces meaningless split thresholds without a lookup table, which defeats the point of choosing a tree for explainability. **Target encoding**: tied with one-hot (~0.59 ROC-AUC) at a fifth of the features, and — specific to a binary target — its split thresholds read directly as "this category's historical win rate," no lookup table needed.

## Why depth control matters (concretely)

An unconstrained tree here grows to depth ~29 with 650+ leaves, memorizing the training set (train ROC-AUC ~0.8-1.0) while generalizing barely above random (test ROC-AUC ~0.55) — worse than logistic regression. Going deep doesn't even buy test performance on its own terms: a depth-capped search (max depth 6, for interpretability) and an uncapped one land on statistically the same test ROC-AUC (~0.59), just with 26 leaves vs. 100+. Tuning below goes straight to the depth-capped search rather than re-demonstrating both halves of that comparison.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', NUMERIC_FEATURES),
    ('bin', 'passthrough', select_binary_features),
    ('cat', TargetEncoder(target_type='binary', cv=5, smooth='auto', random_state=RANDOM_STATE), CATEGORICAL_FEATURES)
])

tree_pipe = Pipeline([
    ('cleaner', QuotationCleaner(rare_walltype_pct=RARE_WALLTYPE_PCT)),
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

param_grid = {
    'model__max_depth': [2, 3, 4, 5, 6],
    'model__min_samples_leaf': [10, 20, 40, 80],
    'model__criterion': ['gini', 'entropy'],
    'model__class_weight': [None, 'balanced'],
}
tscv = TimeSeriesSplit(n_splits=5)
grid_search = GridSearchCV(tree_pipe, param_grid, scoring='average_precision', cv=tscv, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_tree = grid_search.best_estimator_
print('Best params (depth capped at 6):', grid_search.best_params_)
print('Best CV PR-AUC:', round(grid_search.best_score_, 3))
print(f"Tree: depth={best_tree.named_steps['model'].get_depth()}, leaves={best_tree.named_steps['model'].get_n_leaves()}")


## Evaluation

In [ ]:
y_pred = best_tree.predict(X_test)
y_proba = best_tree.predict_proba(X_test)[:, 1]

print('=== Test set performance ===')
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}   (naive 'always Loss' baseline: {1-y_test.mean():.3f})")
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall   : {recall_score(y_test, y_pred):.3f}')
print(f'F1       : {f1_score(y_test, y_pred):.3f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}')
print(f'PR-AUC   : {average_precision_score(y_test, y_proba):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Loss', 'Won']))


If `Precision`/`Recall`/`F1` for "Won" come out near `0` — same pattern as notebook 1 when `class_weight=None` wins under `average_precision` tuning: ROC-AUC/PR-AUC (ranking quality) stay meaningful even when the default 0.5-threshold classification doesn't.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Loss','Won'], yticklabels=['Loss','Won'], ax=axes[0])
axes[0].set_title('Confusion matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color='#27ae60')
axes[1].plot([0,1],[0,1],'--',color='gray'); axes[1].set_title('ROC curve')
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2], color='#16a085')
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='baseline'); axes[2].set_title('Precision-Recall curve'); axes[2].legend()
plt.tight_layout(); plt.show()


## The tree itself

`Client_Clean`/`Suburb`/`Priced_By` are target-encoded — read a low value as "this category's smoothed historical win rate is low," not as an arbitrary code.

In [ ]:
feature_names = best_tree.named_steps['preprocessor'].get_feature_names_out()
clean_names = [f.replace('num__','').replace('bin__','').replace('cat__','') for f in feature_names]

fig, ax = plt.subplots(figsize=(22, 11))
plot_tree(best_tree.named_steps['model'], feature_names=clean_names, class_names=['Loss', 'Won'],
          filled=True, rounded=True, fontsize=8, ax=ax)
plt.tight_layout(); plt.show()


In [ ]:
print(export_text(best_tree.named_steps['model'], feature_names=list(clean_names), max_depth=4))


## Feature Importance & SHAP

In [ ]:
importances = pd.Series(best_tree.named_steps['model'].feature_importances_, index=clean_names).sort_values(ascending=False)
importances = importances[importances > 0]

fig, ax = plt.subplots(figsize=(8, max(3, len(importances) * 0.4)))
ax.barh(importances.index, importances.values, color='#27ae60')
ax.set_xlabel('Importance (impurity reduction)'); ax.set_title('Feature importance — Decision Tree')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
importances


If `Quote_Year` and `Priced_By` dominate here the way `Quote_Year`/`Priced_By_Unknown_Estimator` did in notebook 1's SHAP output, that's a cross-check, not a coincidence — two structurally different models agreeing is what makes an explanation trustworthy.

In [ ]:
import shap

X_train_enc = best_tree[:-1].transform(X_train)
X_test_enc = best_tree[:-1].transform(X_test)

explainer = shap.TreeExplainer(best_tree.named_steps['model'])
raw_shap = explainer(X_test_enc)
shap_values = raw_shap[:, :, 1] if raw_shap.values.ndim == 3 else raw_shap
shap_values.feature_names = clean_names

shap.summary_plot(shap_values, X_test_enc, feature_names=clean_names, max_display=15, show=True)


In [ ]:
test_idx_won = np.where(y_test.values == 1)[0]
example_idx = test_idx_won[np.argmax(y_proba[test_idx_won])]
print(f'Example: test row {example_idx}, actual = Won, predicted probability = {y_proba[example_idx]:.3f}')
shap.plots.waterfall(shap_values[example_idx], max_display=12)


## Model Calibration

A tree this size can only output as many distinct probabilities as it has leaves.

In [ ]:
print(f'Distinct predicted probabilities on the test set: {len(np.unique(y_proba))} (out of {len(y_proba)}, across {best_tree.named_steps["model"].get_n_leaves()} leaves)')


In [ ]:
calib_split = int(split_idx * 0.8)
X_cfit, y_cfit = X.iloc[:calib_split], y.iloc[:calib_split]
X_calib, y_calib = X.iloc[calib_split:split_idx], y.iloc[calib_split:split_idx]

calib_base = clone(best_tree)
calib_base.fit(X_cfit, y_cfit)
proba_uncalibrated = calib_base.predict_proba(X_test)[:, 1]

calibrated_tree = CalibratedClassifierCV(FrozenEstimator(calib_base), method='sigmoid')
calibrated_tree.fit(X_calib, y_calib)
proba_calibrated = calibrated_tree.predict_proba(X_test)[:, 1]

print(f'Distinct probabilities -- uncalibrated: {len(np.unique(proba_uncalibrated))}, calibrated: {len(np.unique(proba_calibrated))}')
print(f'Brier, uncalibrated: {brier_score_loss(y_test, proba_uncalibrated):.4f}  | calibrated: {brier_score_loss(y_test, proba_calibrated):.4f}')
print(f'ROC-AUC, uncalibrated: {roc_auc_score(y_test, proba_uncalibrated):.3f}  | calibrated: {roc_auc_score(y_test, proba_calibrated):.3f}')


In [ ]:
frac_pos_u, mean_pred_u = calibration_curve(y_test, proba_uncalibrated, n_bins=6, strategy='quantile')
frac_pos_c, mean_pred_c = calibration_curve(y_test, proba_calibrated, n_bins=6, strategy='quantile')

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly calibrated')
ax.plot(mean_pred_u, frac_pos_u, marker='o', color='#c0392b', label='Uncalibrated')
ax.plot(mean_pred_c, frac_pos_c, marker='o', color='#27ae60', label='Calibrated')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed win rate'); ax.legend()
plt.tight_layout(); plt.show()


## Proof: fresh-process reload on raw data

Same check as notebook 1, since this is the part that actually broke there the first time — save, then load in a genuinely separate process and predict on an untouched raw row.

In [ ]:
import joblib
joblib.dump(best_tree, 'decision_tree_pipeline.joblib')
joblib.dump(calibrated_tree, 'decision_tree_calibrated.joblib')
print('Saved. quotation_pipeline_utils.py must stay in this folder.')


In [ ]:
import subprocess, textwrap

fresh_process_script = textwrap.dedent("""
    import joblib, pandas as pd
    model = joblib.load('decision_tree_pipeline.joblib')
    raw = pd.read_excel('DATA_PATH_PLACEHOLDER', sheet_name='Data').sample(3, random_state=7)
    print('LOADED OK in a fresh process:', type(model).__name__)
    print('Predictions:', list(model.predict(raw)))
    print('Win probabilities:', list(model.predict_proba(raw)[:, 1].round(3)))
""").replace('DATA_PATH_PLACEHOLDER', DATA_PATH)

with open('_fresh_process_check_dt.py', 'w') as f:
    f.write(fresh_process_script)

result = subprocess.run(['python', '_fresh_process_check_dt.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('FAILED:'); print(result.stderr)


## Summary

- Same shared `quotation_pipeline_utils.py` as notebook 1 — no cleaning logic duplicated, no risk of the two notebooks drifting apart.
- Target encoding chosen over one-hot/ordinal for the same reasons as before: comparable performance, far fewer features, directly-readable split thresholds.
- Depth capped at 6 for interpretability costs nothing in test performance versus letting it go deep — confirmed via direct comparison in the original version of this analysis.
- Same dominant drivers as notebook 1 (`Quote_Year`, `Priced_By`), now from a structurally different model — reinforces that this is a real pattern, not an artifact of one algorithm.
- Slightly weaker ROC-AUC/PR-AUC than tuned logistic regression — expected for a single tree, and the value here is the visual/rule-based explanation, not a leaderboard win.

**Next: Random Forest**, same shared cleaner, same one-`Pipeline` pattern.